# DuckDB & Parquet - Row Groups

Row groups are used by **predicate pushdown** to limit the data to scan, and perform efficient **HTTP range requests** (as seen in the notebook `sql/06-duckdb_parquet-hive_partitioning-s3/duckdb_parquet-hive_partitioning-s3.sql`).

In [1]:
%config SqlMagic.displaylimit = 100
%config SqlMagic.autopandas = False

In [2]:
%%capture
%load_ext sql

In [3]:
# https://duckdb.org/docs/lts/guides/python/jupyter
%sql duckdb:///:memory: --alias duckdb

Connecting to 'duckdb'

Create a sample Parquet file, with a `ROW_GROUP_SIZE` smaller than the total number of rows, so that multiple row groups are generated.

In [4]:
%%sql

COPY (
    SELECT
		i AS number,
		chr((97 + random()* 10_000 % (122 - 97))::INTEGER) AS code,
	FROM range(100_000) tbl(i)
    -- Ordered by code to limit the number of rows on which a given code will by dispatched
    ORDER BY code
)
TO '../data/sample.parquet' (
    FORMAT parquet,
    OVERWRITE true,
    ROW_GROUP_SIZE 5_000
);

Running query in 'duckdb'

Count


In [5]:
%%sql

FROM parquet_schema('../data/sample.parquet');

Running query in 'duckdb'

file_name,name,type,type_length,repetition_type,num_children,converted_type,scale,precision,field_id,logical_type,duckdb_type,column_id
../data/sample.parquet,duckdb_schema,None,None,REQUIRED,2,None,None,None,None,None,None,0
../data/sample.parquet,number,INT64,None,OPTIONAL,None,INT_64,None,None,None,None,BIGINT,1
../data/sample.parquet,code,BYTE_ARRAY,None,OPTIONAL,None,UTF8,None,None,None,None,VARCHAR,2


Check the number of groups (`row_group_id`):

In [6]:
%%sql

SELECT count(DISTINCT row_group_id) AS row_groups
FROM parquet_metadata('../data/sample.parquet'); 

Running query in 'duckdb'

row_groups
17


In [7]:
%%sql

SELECT
    * EXCLUDE(file_name),
FROM parquet_metadata('../data/sample.parquet');

Running query in 'duckdb'

row_group_id,row_group_num_rows,row_group_num_columns,row_group_bytes,column_id,file_offset,num_values,path_in_schema,type,stats_min,stats_max,stats_null_count,stats_distinct_count,stats_min_value,stats_max_value,compression,encodings,index_page_offset,dictionary_page_offset,data_page_offset,total_compressed_size,total_uncompressed_size,key_value_metadata,bloom_filter_offset,bloom_filter_length,min_is_exact,max_is_exact,row_group_compressed_bytes,geo_bbox,geo_types,encoding_stats,stats_nan_count
0,6144,2,49244,0,0,6144,number,INT64,30,99938,0,None,30,99938,SNAPPY,PLAIN,None,None,4,25701,49181,{},None,None,True,True,25768,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",None
0,6144,2,49244,1,0,6144,code,BYTE_ARRAY,a,c,0,3,a,c,SNAPPY,PLAIN_DICTIONARY,None,25705,25735,67,63,{},417445,47,True,True,25768,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None
1,6144,2,49236,0,0,6144,number,INT64,19,99978,0,None,19,99978,SNAPPY,PLAIN,None,None,25772,25440,49181,{},None,None,True,True,25499,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",None
1,6144,2,49236,1,0,6144,code,BYTE_ARRAY,c,d,0,2,c,d,SNAPPY,PLAIN_DICTIONARY,None,51212,51237,59,55,{},417492,47,True,True,25499,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None
2,6144,2,49244,0,0,6144,number,INT64,5,99992,0,None,5,99992,SNAPPY,PLAIN,None,None,51271,25641,49181,{},None,None,True,True,25708,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",None
2,6144,2,49244,1,0,6144,code,BYTE_ARRAY,d,f,0,3,d,f,SNAPPY,PLAIN_DICTIONARY,None,76912,76942,67,63,{},417539,47,True,True,25708,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None
3,6144,2,49236,0,0,6144,number,INT64,11,99969,0,None,11,99969,SNAPPY,PLAIN,None,None,76979,25454,49181,{},None,None,True,True,25513,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",None
3,6144,2,49236,1,0,6144,code,BYTE_ARRAY,f,g,0,2,f,g,SNAPPY,PLAIN_DICTIONARY,None,102433,102458,59,55,{},417586,47,True,True,25513,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None
4,6144,2,49244,0,0,6144,number,INT64,10,99963,0,None,10,99963,SNAPPY,PLAIN,None,None,102492,25628,49181,{},None,None,True,True,25695,None,None,"[{'page_type': 'DATA_PAGE', 'encoding': 'PLAIN', 'count': 1}]",None
4,6144,2,49244,1,0,6144,code,BYTE_ARRAY,g,i,0,3,g,i,SNAPPY,PLAIN_DICTIONARY,None,128120,128150,67,63,{},417633,47,True,True,25695,None,None,"[{'page_type': 'DICTIONARY_PAGE', 'encoding': 'PLAIN', 'count': 1}, {'page_type': 'DATA_PAGE', 'encoding': 'PLAIN_DICTIONARY', 'count': 1}]",None


For every group, the `stats_*` fields have different values.

The field `data_page_offset` indicates the position of the first page of a columnar storage in a row group for a given field.  
This will be used with HTTP range requests.

In [8]:
%%sql

SELECT
    row_group_id,
    path_in_schema,
    data_page_offset,
    stats_min,
    stats_max,
    -- To get all the "stats_" fields
    -- COLUMNS('stats_.+'),
FROM parquet_metadata('../data/sample.parquet')
WHERE path_in_schema = 'code';

Running query in 'duckdb'

row_group_id,path_in_schema,data_page_offset,stats_min,stats_max
0,code,25735,a,c
1,code,51237,c,d
2,code,76942,d,f
3,code,102458,f,g
4,code,128150,g,i
5,code,153717,i,j
6,code,179369,j,l
7,code,204962,l,m
8,code,230596,m,o
9,code,256249,o,p
